# 03 — Preprocessing and feature engineering

## Objective

This notebook converts the exploratory findings into reusable and
leakage-safe preprocessing components.

The main objectives are to:

- reproduce the protected development/holdout split;
- implement deterministic feature engineering;
- handle numerical and categorical missing values;
- encode categorical predictors;
- scale numerical features when required;
- remove deterministic redundancies;
- build reusable scikit-learn preprocessing pipelines;
- validate the transformations before model fitting.

The holdout set is not used to fit, select or tune any transformation.
All data-dependent preprocessing steps must be learned exclusively from
the training folds.

## Import


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split


RANDOM_STATE = 42
HOLDOUT_SIZE = 0.20

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 100)

In [4]:
current_directory = Path.cwd().resolve()

if current_directory.name == "notebooks":
    PROJECT_ROOT = current_directory.parent
else:
    PROJECT_ROOT = current_directory

APPLICATION_TRAIN_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "application_train.csv"
)

application_train = pd.read_csv(
    APPLICATION_TRAIN_PATH
)

application_train.shape

(307511, 122)

In [6]:
required_columns = {
    "SK_ID_CURR",
    "TARGET",
}

missing_required_columns = (
    required_columns
    - set(application_train.columns)
)

assert not missing_required_columns, (
    "Missing required columns: "
    f"{sorted(missing_required_columns)}"
)

source_data_checks = pd.Series(
    {
        "target_is_complete": (
            application_train["TARGET"]
            .notna()
            .all()
        ),
        "target_is_binary": (
            set(
                application_train["TARGET"]
                .unique()
            )
            <= {0, 1}
        ),
        "identifier_is_complete": (
            application_train["SK_ID_CURR"]
            .notna()
            .all()
        ),
        "identifier_is_unique": (
            application_train["SK_ID_CURR"]
            .is_unique
        ),
    },
    name="passed",
)

source_data_checks

target_is_complete        True
target_is_binary          True
identifier_is_complete    True
identifier_is_unique      True
Name: passed, dtype: bool

In [7]:
development_df, holdout_df = train_test_split(
    application_train,
    test_size=HOLDOUT_SIZE,
    random_state=RANDOM_STATE,
    stratify=application_train["TARGET"],
)

development_df = (
    development_df
    .copy()
    .sort_index()
)

holdout_df = (
    holdout_df
    .copy()
    .sort_index()
)

In [8]:
split_audit = pd.DataFrame(
    {
        "full_data": {
            "row_count": len(application_train),
            "target_1_count": (
                application_train["TARGET"]
                .sum()
            ),
            "target_1_rate_pct": (
                application_train["TARGET"]
                .mean()
                * 100
            ),
        },
        "development": {
            "row_count": len(development_df),
            "target_1_count": (
                development_df["TARGET"]
                .sum()
            ),
            "target_1_rate_pct": (
                development_df["TARGET"]
                .mean()
                * 100
            ),
        },
        "holdout": {
            "row_count": len(holdout_df),
            "target_1_count": (
                holdout_df["TARGET"]
                .sum()
            ),
            "target_1_rate_pct": (
                holdout_df["TARGET"]
                .mean()
                * 100
            ),
        },
    }
).T

split_audit.round(4)

,row_count,target_1_count,target_1_rate_pct
full_data,307511.0,24825.0,8.0729
development,246008.0,19860.0,8.0729
holdout,61503.0,4965.0,8.0728


In [9]:
development_ids = set(
    development_df["SK_ID_CURR"]
)

holdout_ids = set(
    holdout_df["SK_ID_CURR"]
)

split_validation_checks = pd.Series(
    {
        "row_counts_reconcile": (
            len(development_df)
            + len(holdout_df)
            == len(application_train)
        ),
        "development_and_holdout_ids_disjoint": (
            development_ids.isdisjoint(
                holdout_ids
            )
        ),
        "development_target_complete": (
            development_df["TARGET"]
            .notna()
            .all()
        ),
        "holdout_target_complete": (
            holdout_df["TARGET"]
            .notna()
            .all()
        ),
        "development_id_unique": (
            development_df["SK_ID_CURR"]
            .is_unique
        ),
        "holdout_id_unique": (
            holdout_df["SK_ID_CURR"]
            .is_unique
        ),
    },
    name="passed",
)

split_validation_checks

row_counts_reconcile                    True
development_and_holdout_ids_disjoint    True
development_target_complete             True
holdout_target_complete                 True
development_id_unique                   True
holdout_id_unique                       True
Name: passed, dtype: bool

## 2.Predictor matrices and feature schema

This section separates identifiers, targets and raw predictors, then builds an
explicit schema for the numerical and categorical features.

The development set remains the only dataset available for exploratory schema
decisions. The holdout predictors are retained solely for final transformation
and evaluation after model selection.

In [12]:
IDENTIFIER_COLUMN = "SK_ID_CURR"
TARGET_COLUMN = "TARGET"

NON_PREDICTOR_COLUMNS = [
    IDENTIFIER_COLUMN,
    TARGET_COLUMN,
]

development_ids = (
    development_df[IDENTIFIER_COLUMN]
    .copy()
)

holdout_ids = (
    holdout_df[IDENTIFIER_COLUMN]
    .copy()
)

y_development = (
    development_df[TARGET_COLUMN]
    .astype("int8")
    .copy()
)

y_holdout = (
    holdout_df[TARGET_COLUMN]
    .astype("int8")
    .copy()
)

X_development_raw = (
    development_df
    .drop(columns=NON_PREDICTOR_COLUMNS)
    .copy()
)

X_holdout_raw = (
    holdout_df
    .drop(columns=NON_PREDICTOR_COLUMNS)
    .copy()
)
matrix_shape_audit = pd.DataFrame(
    {
        "rows": {
            "X_development_raw": (
                X_development_raw.shape[0]
            ),
            "y_development": len(y_development),
            "development_ids": len(development_ids),
            "X_holdout_raw": (
                X_holdout_raw.shape[0]
            ),
            "y_holdout": len(y_holdout),
            "holdout_ids": len(holdout_ids),
        },
        "columns": {
            "X_development_raw": (
                X_development_raw.shape[1]
            ),
            "y_development": 1,
            "development_ids": 1,
            "X_holdout_raw": (
                X_holdout_raw.shape[1]
            ),
            "y_holdout": 1,
            "holdout_ids": 1,
        },
    }
)

matrix_shape_audit

,rows,columns
X_development_raw,246008,120
y_development,246008,1
development_ids,246008,1
X_holdout_raw,61503,120
y_holdout,61503,1
holdout_ids,61503,1


In [14]:
raw_categorical_features = [
    feature
    for feature in X_development_raw.columns
    if (
        pd.api.types.is_object_dtype(
            X_development_raw[feature]
        )
        or pd.api.types.is_string_dtype(
            X_development_raw[feature]
        )
        or isinstance(
            X_development_raw[feature].dtype,
            pd.CategoricalDtype,
        )
    )
]

raw_numerical_features = [
    feature
    for feature in X_development_raw.columns
    if pd.api.types.is_numeric_dtype(
        X_development_raw[feature]
    )
]

classified_features = set(
    raw_categorical_features
    + raw_numerical_features
)

unsupported_features = [
    feature
    for feature in X_development_raw.columns
    if feature not in classified_features
]

unsupported_features

[]

In [15]:
raw_feature_type_counts = pd.Series(
    {
        "raw_predictors": (
            X_development_raw.shape[1]
        ),
        "raw_numerical_features": len(
            raw_numerical_features
        ),
        "raw_categorical_features": len(
            raw_categorical_features
        ),
        "unsupported_features": len(
            unsupported_features
        ),
    },
    name="count",
)

raw_feature_type_counts

raw_predictors              120
raw_numerical_features      104
raw_categorical_features     16
unsupported_features          0
Name: count, dtype: int64

In [18]:
def build_raw_feature_schema(
    df: pd.DataFrame,
    categorical_features: list[str],
) -> pd.DataFrame:
    categorical_feature_set = set(
        categorical_features
    )

    schema_records = []

    for feature in df.columns:
        series = df[feature]

        feature_kind = (
            "categorical"
            if feature
            in categorical_feature_set
            else "numerical"
        )

        value_frequencies = (
            series.value_counts(
                dropna=False,
                normalize=True,
            )
        )

        most_frequent_share_pct = (
            value_frequencies.iloc[0] * 100
            if not value_frequencies.empty
            else np.nan
        )

        schema_records.append(
            {
                "feature": feature,
                "feature_kind": feature_kind,
                "dtype": str(series.dtype),
                "missing_count": (
                    series.isna().sum()
                ),
                "missing_pct": (
                    series.isna().mean() * 100
                ),
                "n_unique_observed": (
                    series.nunique(dropna=True)
                ),
                "n_unique_with_missing": (
                    series.nunique(dropna=False)
                ),
                "most_frequent_share_pct": (
                    most_frequent_share_pct
                ),
                "is_constant": (
                    series.nunique(dropna=True)
                    <= 1
                ),
                "is_near_constant": (
                    most_frequent_share_pct
                    >= 99.5
                ),
            }
        )

    return pd.DataFrame(schema_records)


raw_feature_schema = build_raw_feature_schema(
    X_development_raw,
    raw_categorical_features,
)

raw_feature_schema.head(10)

,feature,feature_kind,dtype,missing_count,missing_pct,n_unique_observed,n_unique_with_missing,most_frequent_share_pct,is_constant,is_near_constant
0,NAME_CONTRACT_TYPE,categorical,object,0,0.000000,2,2,90.468603,False,False
1,CODE_GENDER,categorical,object,0,0.000000,3,3,65.805584,False,False
2,FLAG_OWN_CAR,categorical,object,0,0.000000,2,2,66.019398,False,False
3,FLAG_OWN_REALTY,categorical,object,0,0.000000,2,2,69.383922,False,False
4,CNT_CHILDREN,numerical,int64,0,0.000000,15,15,70.010731,False,False
5,AMT_INCOME_TOTAL,numerical,float64,0,0.000000,2168,2168,11.658970,False,False
6,AMT_CREDIT,numerical,float64,0,0.000000,5274,5274,3.177945,False,False
7,AMT_ANNUITY,numerical,float64,10,0.004065,13135,13136,2.096273,False,False
8,AMT_GOODS_PRICE,numerical,float64,221,0.089834,884,885,8.453790,False,False
9,NAME_TYPE_SUITE,categorical,object,1029,0.418279,7,8,80.813632,False,False


In [19]:
schema_summary = (
    raw_feature_schema
    .groupby(
        "feature_kind",
        observed=True,
    )
    .agg(
        feature_count=("feature", "count"),
        features_with_missing=(
            "missing_count",
            lambda values: (values > 0).sum(),
        ),
        median_missing_pct=(
            "missing_pct",
            "median",
        ),
        maximum_missing_pct=(
            "missing_pct",
            "max",
        ),
        constant_feature_count=(
            "is_constant",
            "sum",
        ),
        near_constant_feature_count=(
            "is_near_constant",
            "sum",
        ),
    )
)

schema_summary.round(3)

,feature_count,features_with_missing,median_missing_pct,maximum_missing_pct,constant_feature_count,near_constant_feature_count
feature_kind,,,,,,
categorical,16,6,0.000,68.378,0,0
numerical,104,61,6.922,69.840,0,16


In [21]:
(
    raw_feature_schema
    .sort_values(
        "missing_pct",
        ascending=False,
    )
    .head(20)
)

,feature,feature_kind,dtype,missing_count,missing_pct,n_unique_observed,n_unique_with_missing,most_frequent_share_pct,is_constant,is_near_constant
60,COMMONAREA_MODE,numerical,float64,171811,69.839599,2988,2989,69.839599,False,False
74,COMMONAREA_MEDI,numerical,float64,171811,69.839599,3060,3061,69.839599,False,False
46,COMMONAREA_AVG,numerical,float64,171811,69.839599,3047,3048,69.839599,False,False
82,NONLIVINGAPARTMENTS_MEDI,numerical,float64,170729,69.399776,202,203,69.399776,False,False
54,NONLIVINGAPARTMENTS_AVG,numerical,float64,170729,69.399776,364,365,69.399776,False,False
68,NONLIVINGAPARTMENTS_MODE,numerical,float64,170729,69.399776,157,158,69.399776,False,False
84,FONDKAPREMONT_MODE,categorical,object,168215,68.377858,4,5,68.377858,False,False
52,LIVINGAPARTMENTS_AVG,numerical,float64,168119,68.338835,1810,1811,68.338835,False,False
66,LIVINGAPARTMENTS_MODE,numerical,float64,168119,68.338835,725,726,68.338835,False,False
80,LIVINGAPARTMENTS_MEDI,numerical,float64,168119,68.338835,1079,1080,68.338835,False,False


In [22]:
low_variability_features = (
    raw_feature_schema.loc[
        raw_feature_schema[
            [
                "is_constant",
                "is_near_constant",
            ]
        ].any(axis=1)
    ]
    .sort_values(
        "most_frequent_share_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)

low_variability_features

,feature,feature_kind,dtype,missing_count,missing_pct,n_unique_observed,n_unique_with_missing,most_frequent_share_pct,is_constant,is_near_constant
0,FLAG_MOBIL,numerical,int64,0,0.0,2,2,99.999594,False,True
1,FLAG_DOCUMENT_12,numerical,int64,0,0.0,2,2,99.999594,False,True
2,FLAG_DOCUMENT_10,numerical,int64,0,0.0,2,2,99.998374,False,True
3,FLAG_DOCUMENT_2,numerical,int64,0,0.0,2,2,99.995529,False,True
4,FLAG_DOCUMENT_4,numerical,int64,0,0.0,2,2,99.991057,False,True
5,FLAG_DOCUMENT_7,numerical,int64,0,0.0,2,2,99.981301,False,True
6,FLAG_DOCUMENT_17,numerical,int64,0,0.0,2,2,99.975204,False,True
7,FLAG_DOCUMENT_21,numerical,int64,0,0.0,2,2,99.967887,False,True
8,FLAG_DOCUMENT_20,numerical,int64,0,0.0,2,2,99.949595,False,True
9,FLAG_DOCUMENT_19,numerical,int64,0,0.0,2,2,99.939026,False,True


In [23]:
low_cardinality_numerical_features = (
    raw_feature_schema.loc[
        (
            raw_feature_schema[
                "feature_kind"
            ]
            == "numerical"
        )
        & (
            raw_feature_schema[
                "n_unique_observed"
            ]
            <= 10
        ),
        [
            "feature",
            "dtype",
            "n_unique_observed",
            "missing_pct",
            "most_frequent_share_pct",
        ],
    ]
    .sort_values(
        [
            "n_unique_observed",
            "feature",
        ]
    )
    .reset_index(drop=True)
)

low_cardinality_numerical_features

,feature,dtype,n_unique_observed,missing_pct,most_frequent_share_pct
0,FLAG_CONT_MOBILE,int64,2,0.000000,99.812201
1,FLAG_DOCUMENT_10,int64,2,0.000000,99.998374
2,FLAG_DOCUMENT_11,int64,2,0.000000,99.613427
3,FLAG_DOCUMENT_12,int64,2,0.000000,99.999594
4,FLAG_DOCUMENT_13,int64,2,0.000000,99.661800
5,FLAG_DOCUMENT_14,int64,2,0.000000,99.704888
6,FLAG_DOCUMENT_15,int64,2,0.000000,99.880085
7,FLAG_DOCUMENT_16,int64,2,0.000000,98.998000
8,FLAG_DOCUMENT_17,int64,2,0.000000,99.975204
9,FLAG_DOCUMENT_18,int64,2,0.000000,99.186205


In [24]:
development_dtype_map = (
    X_development_raw.dtypes
    .astype(str)
)

holdout_dtype_map = (
    X_holdout_raw.dtypes
    .astype(str)
)

dtype_comparison = pd.DataFrame(
    {
        "development_dtype": (
            development_dtype_map
        ),
        "holdout_dtype": (
            holdout_dtype_map
        ),
    }
)

dtype_comparison[
    "dtype_matches"
] = (
    dtype_comparison[
        "development_dtype"
    ]
    == dtype_comparison[
        "holdout_dtype"
    ]
)

dtype_comparison.loc[
    ~dtype_comparison["dtype_matches"]
]

,development_dtype,holdout_dtype,dtype_matches


In [25]:
feature_schema_checks = pd.Series(
    {
        "target_excluded_from_development_X": (
            TARGET_COLUMN
            not in X_development_raw.columns
        ),
        "identifier_excluded_from_development_X": (
            IDENTIFIER_COLUMN
            not in X_development_raw.columns
        ),
        "target_excluded_from_holdout_X": (
            TARGET_COLUMN
            not in X_holdout_raw.columns
        ),
        "identifier_excluded_from_holdout_X": (
            IDENTIFIER_COLUMN
            not in X_holdout_raw.columns
        ),
        "development_X_y_indices_match": (
            X_development_raw.index.equals(
                y_development.index
            )
        ),
        "holdout_X_y_indices_match": (
            X_holdout_raw.index.equals(
                y_holdout.index
            )
        ),
        "development_X_id_indices_match": (
            X_development_raw.index.equals(
                development_ids.index
            )
        ),
        "holdout_X_id_indices_match": (
            X_holdout_raw.index.equals(
                holdout_ids.index
            )
        ),
        "development_holdout_columns_match": (
            X_development_raw.columns.equals(
                X_holdout_raw.columns
            )
        ),
        "development_holdout_dtypes_match": (
            dtype_comparison[
                "dtype_matches"
            ].all()
        ),
        "all_features_classified_once": (
            len(raw_categorical_features)
            + len(raw_numerical_features)
            == X_development_raw.shape[1]
        ),
        "no_unsupported_feature_dtype": (
            len(unsupported_features) == 0
        ),
        "development_target_is_binary": (
            set(y_development.unique())
            <= {0, 1}
        ),
        "holdout_target_is_binary": (
            set(y_holdout.unique())
            <= {0, 1}
        ),
    },
    name="passed",
)

feature_schema_checks

target_excluded_from_development_X        True
identifier_excluded_from_development_X    True
target_excluded_from_holdout_X            True
identifier_excluded_from_holdout_X        True
development_X_y_indices_match             True
holdout_X_y_indices_match                 True
development_X_id_indices_match            True
holdout_X_id_indices_match                True
development_holdout_columns_match         True
development_holdout_dtypes_match          True
all_features_classified_once              True
no_unsupported_feature_dtype              True
development_target_is_binary              True
holdout_target_is_binary                  True
Name: passed, dtype: bool

In [26]:
RAW_CATEGORICAL_FEATURES = tuple(
    raw_categorical_features
)

RAW_NUMERICAL_FEATURES = tuple(
    raw_numerical_features
)

RAW_PREDICTOR_FEATURES = tuple(
    X_development_raw.columns
)

len(RAW_PREDICTOR_FEATURES), (
    len(RAW_NUMERICAL_FEATURES),
    len(RAW_CATEGORICAL_FEATURES),
)

(120, (104, 16))

## Raw feature-schema findings

- The identifier and target have been separated from the predictor matrices.
- The raw application table contains 120 candidate predictors: 104 numerical
  features and 16 categorical features.
- Development and holdout predictor matrices have identical column order and
  data types.
- Several numerical predictors are binary indicators or low-cardinality counts;
  they remain numerical at this stage.
- Missing values and special categorical labels have not yet been modified.
- No feature-selection, imputation, encoding or scaling parameter has been
  estimated.
- All schema decisions were derived from the development data. The holdout set
  remains unavailable for feature or model selection.

# 3.Deterministic feature engineering

This section implements deterministic application-level feature engineering as a
reusable scikit-learn transformer.

The transformations:

- do not use the target;
- do not estimate statistics from the data;
- preserve row order and applicant indices;
- handle the `DAYS_EMPLOYED = 365243` sentinel explicitly;
- reproduce the duration, external-score availability and financial-ratio
  features studied during EDA;
- return missing values rather than infinities when a ratio is undefined.

The transformer will later be placed at the beginning of each modeling pipeline.

In [27]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

In [28]:
def safe_divide(
    numerator: pd.Series,
    denominator: pd.Series,
) -> pd.Series:
    """
    Divide two numerical Series safely.

    Undefined ratios, non-positive denominators and infinite
    results are represented by NaN.
    """
    numerator = pd.to_numeric(
        numerator,
        errors="coerce",
    ).astype("float64")

    denominator = pd.to_numeric(
        denominator,
        errors="coerce",
    ).astype("float64")

    valid_denominator = (
        denominator.notna()
        & denominator.gt(0)
    )

    result = pd.Series(
        np.nan,
        index=numerator.index,
        dtype="float64",
    )

    result.loc[valid_denominator] = (
        numerator.loc[valid_denominator]
        / denominator.loc[valid_denominator]
    )

    return result.replace(
        [np.inf, -np.inf],
        np.nan,
    )

In [30]:
ENGINEERED_FEATURES = (
    "AGE_YEARS",
    "EMPLOYMENT_YEARS",
    "DAYS_EMPLOYED_ANOMALOUS",
    "REGISTRATION_YEARS",
    "ID_PUBLISH_YEARS",
    "EMPLOYMENT_START_AGE",
    "EXT_SOURCE_AVAILABLE_COUNT",
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "CREDIT_GOODS_RATIO",
    "ANNUITY_CREDIT_RATIO",
    "CREDIT_TERM_PROXY",
    "INCOME_PER_PERSON",
    "EMPLOYED_AGE_RATIO",
)

RATIO_FEATURES = (
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "CREDIT_GOODS_RATIO",
    "ANNUITY_CREDIT_RATIO",
    "CREDIT_TERM_PROXY",
    "INCOME_PER_PERSON",
    "EMPLOYED_AGE_RATIO",
)

len(ENGINEERED_FEATURES), len(RATIO_FEATURES)

(14, 7)

In [31]:
class ApplicationFeatureEngineer(
    BaseEstimator,
    TransformerMixin,
):
    """
    Create deterministic application-level features.

    No target-dependent or data-dependent statistics are learned.
    """

    REQUIRED_COLUMNS = (
        "DAYS_BIRTH",
        "DAYS_EMPLOYED",
        "DAYS_REGISTRATION",
        "DAYS_ID_PUBLISH",
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3",
        "AMT_INCOME_TOTAL",
        "AMT_CREDIT",
        "AMT_ANNUITY",
        "AMT_GOODS_PRICE",
        "CNT_FAM_MEMBERS",
    )

    EXT_SOURCE_COLUMNS = (
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3",
    )

    def __init__(
        self,
        days_per_year: float = 365.25,
        employed_sentinel: int = 365243,
    ):
        self.days_per_year = days_per_year
        self.employed_sentinel = employed_sentinel

    def _validate_dataframe(
        self,
        X: pd.DataFrame,
    ) -> None:
        if not isinstance(X, pd.DataFrame):
            raise TypeError(
                "ApplicationFeatureEngineer expects "
                "a pandas DataFrame."
            )

        missing_columns = (
            set(self.REQUIRED_COLUMNS)
            - set(X.columns)
        )

        if missing_columns:
            raise ValueError(
                "Missing required columns: "
                f"{sorted(missing_columns)}"
            )

        non_numerical_columns = [
            column
            for column in self.REQUIRED_COLUMNS
            if not pd.api.types.is_numeric_dtype(
                X[column]
            )
        ]

        if non_numerical_columns:
            raise TypeError(
                "The following required columns must "
                "be numerical: "
                f"{non_numerical_columns}"
            )

    def fit(
        self,
        X: pd.DataFrame,
        y=None,
    ):
        self._validate_dataframe(X)

        feature_collisions = (
            set(ENGINEERED_FEATURES)
            & set(X.columns)
        )

        if feature_collisions:
            raise ValueError(
                "Engineered-feature names already exist "
                "in the input: "
                f"{sorted(feature_collisions)}"
            )

        self.feature_names_in_ = np.asarray(
            X.columns,
            dtype=object,
        )

        self.n_features_in_ = len(
            self.feature_names_in_
        )

        self.feature_names_out_ = np.asarray(
            [
                *self.feature_names_in_,
                *ENGINEERED_FEATURES,
            ],
            dtype=object,
        )

        return self

    def transform(
        self,
        X: pd.DataFrame,
    ) -> pd.DataFrame:
        check_is_fitted(
            self,
            attributes=[
                "feature_names_in_",
                "feature_names_out_",
            ],
        )

        self._validate_dataframe(X)

        if not X.columns.equals(
            pd.Index(self.feature_names_in_)
        ):
            raise ValueError(
                "Input columns or their order differ "
                "from the fitted schema."
            )

        transformed = X.copy()

        # Sentinel treatment
        days_employed = transformed[
            "DAYS_EMPLOYED"
        ].astype("float64")

        employed_anomalous = (
            days_employed
            .eq(self.employed_sentinel)
        )

        transformed[
            "DAYS_EMPLOYED_ANOMALOUS"
        ] = employed_anomalous.astype("int8")

        transformed["DAYS_EMPLOYED"] = (
            days_employed.mask(
                employed_anomalous
            )
        )

        # Duration features
        transformed["AGE_YEARS"] = (
            -transformed["DAYS_BIRTH"]
            / self.days_per_year
        )

        transformed["EMPLOYMENT_YEARS"] = (
            -transformed["DAYS_EMPLOYED"]
            / self.days_per_year
        )

        transformed["REGISTRATION_YEARS"] = (
            -transformed["DAYS_REGISTRATION"]
            / self.days_per_year
        )

        transformed["ID_PUBLISH_YEARS"] = (
            -transformed["DAYS_ID_PUBLISH"]
            / self.days_per_year
        )

        transformed["EMPLOYMENT_START_AGE"] = (
            transformed["AGE_YEARS"]
            - transformed["EMPLOYMENT_YEARS"]
        )

        # External-score availability
        transformed[
            "EXT_SOURCE_AVAILABLE_COUNT"
        ] = (
            transformed[
                list(self.EXT_SOURCE_COLUMNS)
            ]
            .notna()
            .sum(axis=1)
            .astype("int8")
        )

        # Financial ratios
        transformed["CREDIT_INCOME_RATIO"] = (
            safe_divide(
                transformed["AMT_CREDIT"],
                transformed["AMT_INCOME_TOTAL"],
            )
        )

        transformed["ANNUITY_INCOME_RATIO"] = (
            safe_divide(
                transformed["AMT_ANNUITY"],
                transformed["AMT_INCOME_TOTAL"],
            )
        )

        transformed["CREDIT_GOODS_RATIO"] = (
            safe_divide(
                transformed["AMT_CREDIT"],
                transformed["AMT_GOODS_PRICE"],
            )
        )

        transformed["ANNUITY_CREDIT_RATIO"] = (
            safe_divide(
                transformed["AMT_ANNUITY"],
                transformed["AMT_CREDIT"],
            )
        )

        transformed["CREDIT_TERM_PROXY"] = (
            safe_divide(
                transformed["AMT_CREDIT"],
                transformed["AMT_ANNUITY"],
            )
        )

        transformed["INCOME_PER_PERSON"] = (
            safe_divide(
                transformed["AMT_INCOME_TOTAL"],
                transformed["CNT_FAM_MEMBERS"],
            )
        )

        transformed["EMPLOYED_AGE_RATIO"] = (
            safe_divide(
                transformed["EMPLOYMENT_YEARS"],
                transformed["AGE_YEARS"],
            )
        )

        return transformed.loc[
            :,
            self.feature_names_out_,
        ]

    def get_feature_names_out(
        self,
        input_features=None,
    ) -> np.ndarray:
        check_is_fitted(
            self,
            attributes=["feature_names_out_"],
        )

        if input_features is not None:
            input_features = np.asarray(
                input_features,
                dtype=object,
            )

            if not np.array_equal(
                input_features,
                self.feature_names_in_,
            ):
                raise ValueError(
                    "input_features does not match "
                    "feature_names_in_."
                )

        return self.feature_names_out_.copy()

In [32]:
feature_engineer = ApplicationFeatureEngineer()

mutation_test_input = (
    X_development_raw
    .head(1_000)
    .copy(deep=True)
)

mutation_test_snapshot = (
    mutation_test_input
    .copy(deep=True)
)

feature_engineer.fit(
    X_development_raw
)

_ = feature_engineer.transform(
    mutation_test_input
)

input_was_not_modified = (
    mutation_test_input.equals(
        mutation_test_snapshot
    )
)

input_was_not_modified

True

In [33]:
X_development_engineered = (
    feature_engineer.transform(
        X_development_raw
    )
)

X_development_raw.shape, (
    X_development_engineered.shape
)

((246008, 120), (246008, 134))

In [34]:
X_development_engineered[
    list(ENGINEERED_FEATURES)
].head()

,AGE_YEARS,EMPLOYMENT_YEARS,DAYS_EMPLOYED_ANOMALOUS,REGISTRATION_YEARS,ID_PUBLISH_YEARS,EMPLOYMENT_START_AGE,EXT_SOURCE_AVAILABLE_COUNT,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,CREDIT_GOODS_RATIO,ANNUITY_CREDIT_RATIO,CREDIT_TERM_PROXY,INCOME_PER_PERSON,EMPLOYED_AGE_RATIO
0,25.902806,1.744011,0,9.987680,5.804244,24.158795,3,2.007889,0.121978,1.158397,0.060749,16.461104,202500.0,0.067329
1,45.900068,3.252567,0,3.247091,0.796715,42.647502,2,4.790750,0.132217,1.145199,0.027598,36.234085,135000.0,0.070862
2,52.145106,0.616016,0,11.663244,6.929500,51.529090,2,2.000000,0.100000,1.000000,0.050000,20.000000,67500.0,0.011814
3,52.032854,8.320329,0,26.921287,6.672142,43.712526,1,2.316167,0.219900,1.052803,0.094941,10.532818,67500.0,0.159905
4,54.570842,8.317591,0,11.802875,9.467488,46.253251,1,4.222222,0.179963,1.000000,0.042623,23.461618,121500.0,0.152418


In [35]:
engineered_feature_audit = pd.DataFrame(
    {
        "dtype": (
            X_development_engineered[
                list(ENGINEERED_FEATURES)
            ]
            .dtypes
            .astype(str)
        ),
        "missing_count": (
            X_development_engineered[
                list(ENGINEERED_FEATURES)
            ]
            .isna()
            .sum()
        ),
        "missing_pct": (
            X_development_engineered[
                list(ENGINEERED_FEATURES)
            ]
            .isna()
            .mean()
            .mul(100)
        ),
        "minimum": (
            X_development_engineered[
                list(ENGINEERED_FEATURES)
            ]
            .min()
        ),
        "median": (
            X_development_engineered[
                list(ENGINEERED_FEATURES)
            ]
            .median()
        ),
        "maximum": (
            X_development_engineered[
                list(ENGINEERED_FEATURES)
            ]
            .max()
        ),
    }
)

engineered_feature_audit.round(3)

,dtype,missing_count,missing_pct,minimum,median,maximum
AGE_YEARS,float64,0,0.000,20.504,43.105,6.907300e+01
EMPLOYMENT_YEARS,float64,44143,17.944,-0.000,4.512,4.904000e+01
DAYS_EMPLOYED_ANOMALOUS,int8,0,0.000,0.000,0.000,1.000000e+00
REGISTRATION_YEARS,float64,0,0.000,-0.000,12.334,6.754800e+01
ID_PUBLISH_YEARS,float64,0,0.000,0.000,8.912,1.970400e+01
EMPLOYMENT_START_AGE,float64,44143,17.944,17.916,32.723,6.816400e+01
EXT_SOURCE_AVAILABLE_COUNT,int8,0,0.000,0.000,2.000,3.000000e+00
CREDIT_INCOME_RATIO,float64,0,0.000,0.005,3.269,4.922700e+01
ANNUITY_INCOME_RATIO,float64,10,0.004,0.000,0.163,1.571000e+00
CREDIT_GOODS_RATIO,float64,221,0.090,0.150,1.119,6.000000e+00


In [36]:
sentinel_audit = pd.Series(
    {
        "raw_sentinel_count": (
            X_development_raw[
                "DAYS_EMPLOYED"
            ]
            .eq(365243)
            .sum()
        ),
        "engineered_anomaly_count": (
            X_development_engineered[
                "DAYS_EMPLOYED_ANOMALOUS"
            ]
            .sum()
        ),
        "remaining_sentinel_count": (
            X_development_engineered[
                "DAYS_EMPLOYED"
            ]
            .eq(365243)
            .sum()
        ),
        "clean_employment_missing_count": (
            X_development_engineered[
                "DAYS_EMPLOYED"
            ]
            .isna()
            .sum()
        ),
    },
    name="count",
)

sentinel_audit

raw_sentinel_count                44143
engineered_anomaly_count          44143
remaining_sentinel_count              0
clean_employment_missing_count    44143
Name: count, dtype: int64

In [37]:
expected_age_years = (
    -X_development_raw["DAYS_BIRTH"]
    / 365.25
)

expected_credit_income_ratio = (
    safe_divide(
        X_development_raw["AMT_CREDIT"],
        X_development_raw[
            "AMT_INCOME_TOTAL"
        ],
    )
)

expected_credit_term_proxy = (
    safe_divide(
        X_development_raw["AMT_CREDIT"],
        X_development_raw["AMT_ANNUITY"],
    )
)

formula_checks = pd.Series(
    {
        "age_formula_matches": (
            np.allclose(
                X_development_engineered[
                    "AGE_YEARS"
                ],
                expected_age_years,
                equal_nan=True,
            )
        ),
        "credit_income_formula_matches": (
            np.allclose(
                X_development_engineered[
                    "CREDIT_INCOME_RATIO"
                ],
                expected_credit_income_ratio,
                equal_nan=True,
            )
        ),
        "credit_term_formula_matches": (
            np.allclose(
                X_development_engineered[
                    "CREDIT_TERM_PROXY"
                ],
                expected_credit_term_proxy,
                equal_nan=True,
            )
        ),
        "employment_start_identity_matches": (
            np.allclose(
                X_development_engineered[
                    "EMPLOYMENT_START_AGE"
                ],
                (
                    X_development_engineered[
                        "AGE_YEARS"
                    ]
                    - X_development_engineered[
                        "EMPLOYMENT_YEARS"
                    ]
                ),
                equal_nan=True,
            )
        ),
    },
    name="passed",
)

formula_checks

age_formula_matches                  True
credit_income_formula_matches        True
credit_term_formula_matches          True
employment_start_identity_matches    True
Name: passed, dtype: bool

In [40]:
engineered_numeric_array = (
    X_development_engineered[
        list(ENGINEERED_FEATURES)
    ]
    .to_numpy(dtype="float64")
)

infinite_engineered_value_count = (
    np.isinf(
        engineered_numeric_array
    ).sum()
)

external_count_values = set(
    X_development_engineered[
        "EXT_SOURCE_AVAILABLE_COUNT"
    ].unique()
)

employment_anomaly_values = set(
    X_development_engineered[
        "DAYS_EMPLOYED_ANOMALOUS"
    ].unique()
)

infinite_engineered_value_count, (
    external_count_values
), employment_anomaly_values

(0, {0, 1, 2, 3}, {0, 1})

In [39]:
expected_output_feature_count = (
    X_development_raw.shape[1]
    + len(ENGINEERED_FEATURES)
)

source_anomaly_indicator = (
    X_development_raw[
        "DAYS_EMPLOYED"
    ]
    .eq(365243)
    .astype("int8")
)

feature_engineering_checks = pd.Series(
    {
        "input_dataframe_not_modified": (
            input_was_not_modified
        ),
        "row_count_preserved": (
            len(X_development_engineered)
            == len(X_development_raw)
        ),
        "index_preserved": (
            X_development_engineered
            .index
            .equals(
                X_development_raw.index
            )
        ),
        "expected_feature_count": (
            X_development_engineered.shape[1]
            == expected_output_feature_count
        ),
        "all_engineered_features_created": (
            set(ENGINEERED_FEATURES)
            <= set(
                X_development_engineered.columns
            )
        ),
        "original_columns_preserved": (
            set(X_development_raw.columns)
            <= set(
                X_development_engineered.columns
            )
        ),
        "target_not_created": (
            TARGET_COLUMN
            not in X_development_engineered.columns
        ),
        "identifier_not_created": (
            IDENTIFIER_COLUMN
            not in X_development_engineered.columns
        ),
        "employment_sentinel_removed": (
            not X_development_engineered[
                "DAYS_EMPLOYED"
            ]
            .eq(365243)
            .any()
        ),
        "anomaly_indicator_matches_source": (
            X_development_engineered[
                "DAYS_EMPLOYED_ANOMALOUS"
            ]
            .equals(
                source_anomaly_indicator
            )
        ),
        "external_count_is_valid": (
            external_count_values
            <= {0, 1, 2, 3}
        ),
        "employment_indicator_is_binary": (
            employment_anomaly_values
            <= {0, 1}
        ),
        "no_infinite_engineered_values": (
            infinite_engineered_value_count
            == 0
        ),
        "all_formula_checks_pass": (
            formula_checks.all()
        ),
    },
    name="passed",
)

feature_engineering_checks

input_dataframe_not_modified        True
row_count_preserved                 True
index_preserved                     True
expected_feature_count              True
all_engineered_features_created     True
original_columns_preserved          True
target_not_created                  True
identifier_not_created              True
employment_sentinel_removed         True
anomaly_indicator_matches_source    True
external_count_is_valid             True
employment_indicator_is_binary      True
no_infinite_engineered_values       True
all_formula_checks_pass             True
Name: passed, dtype: bool

## Deterministic feature-engineering findings

- Four raw day-based variables were converted into interpretable durations using
  365.25 days per year.
- The `DAYS_EMPLOYED = 365243` sentinel was replaced by a missing value and
  preserved through a separate binary anomaly indicator.
- External-score availability was summarized without imputing the external scores.
- Seven financial and demographic ratios were reproduced consistently with the
  exploratory analysis.
- Undefined ratios are represented by missing values rather than positive or
  negative infinity.
- The transformation preserves applicant indices and does not modify the input
  DataFrame.
- No target information or data-dependent statistic is used by the transformer.
- Deterministic duplicates and exact inverse features are still present and will be
  handled explicitly in the next section.
- The holdout set remains isolated.

# 4. Feature selection rules and model-specific views

This section defines deterministic feature-selection rules and separate candidate
feature views for linear and tree-based models.

The rules:

- remove raw duration variables when their cleaned year-based representations are
  retained;
- remove one member of an exact reciprocal feature pair;
- avoid exact linear dependence in the logistic-regression view;
- retain useful interaction features for tree-based models;
- preserve all categorical variables and rare indicators at this stage;
- do not use the target or holdout set to select predictors.

Any later data-dependent feature selection must be fitted inside cross-validation.

In [42]:
COMMON_EXCLUDED_FEATURES = (
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
    "CREDIT_TERM_PROXY",
)

LOGISTIC_SPECIFIC_EXCLUDED_FEATURES = (
    "EMPLOYMENT_START_AGE",
)

TREE_SPECIFIC_EXCLUDED_FEATURES = ()

LOGISTIC_EXCLUDED_FEATURES = (
    *COMMON_EXCLUDED_FEATURES,
    *LOGISTIC_SPECIFIC_EXCLUDED_FEATURES,
)

TREE_EXCLUDED_FEATURES = (
    *COMMON_EXCLUDED_FEATURES,
    *TREE_SPECIFIC_EXCLUDED_FEATURES,
)

In [43]:
feature_exclusion_rules = pd.DataFrame(
    [
        {
            "excluded_feature": "DAYS_BIRTH",
            "retained_feature": "AGE_YEARS",
            "applies_to": "all_models",
            "reason": (
                "Exact signed rescaling; the year-based "
                "representation is more interpretable."
            ),
        },
        {
            "excluded_feature": "DAYS_EMPLOYED",
            "retained_feature": "EMPLOYMENT_YEARS",
            "applies_to": "all_models",
            "reason": (
                "Exact signed rescaling after sentinel "
                "cleaning."
            ),
        },
        {
            "excluded_feature": "DAYS_REGISTRATION",
            "retained_feature": "REGISTRATION_YEARS",
            "applies_to": "all_models",
            "reason": (
                "Exact signed rescaling; keeping both "
                "adds no information."
            ),
        },
        {
            "excluded_feature": "DAYS_ID_PUBLISH",
            "retained_feature": "ID_PUBLISH_YEARS",
            "applies_to": "all_models",
            "reason": (
                "Exact signed rescaling; keeping both "
                "adds no information."
            ),
        },
        {
            "excluded_feature": "CREDIT_TERM_PROXY",
            "retained_feature": "ANNUITY_CREDIT_RATIO",
            "applies_to": "all_models",
            "reason": (
                "Exact reciprocal when both ratios are "
                "defined."
            ),
        },
        {
            "excluded_feature": "EMPLOYMENT_START_AGE",
            "retained_feature": (
                "AGE_YEARS and EMPLOYMENT_YEARS"
            ),
            "applies_to": "logistic_regression",
            "reason": (
                "Exact linear combination that would "
                "create perfect multicollinearity."
            ),
        },
    ]
)

feature_exclusion_rules

,excluded_feature,retained_feature,applies_to,reason
0,DAYS_BIRTH,AGE_YEARS,all_models,Exact signed rescaling; the year-based represe...
1,DAYS_EMPLOYED,EMPLOYMENT_YEARS,all_models,Exact signed rescaling after sentinel cleaning.
2,DAYS_REGISTRATION,REGISTRATION_YEARS,all_models,Exact signed rescaling; keeping both adds no i...
3,DAYS_ID_PUBLISH,ID_PUBLISH_YEARS,all_models,Exact signed rescaling; keeping both adds no i...
4,CREDIT_TERM_PROXY,ANNUITY_CREDIT_RATIO,all_models,Exact reciprocal when both ratios are defined.
5,EMPLOYMENT_START_AGE,AGE_YEARS and EMPLOYMENT_YEARS,logistic_regression,Exact linear combination that would create per...


In [46]:
engineered_feature_order = tuple(
    X_development_engineered.columns
)

logistic_excluded_set = set(
    LOGISTIC_EXCLUDED_FEATURES
)

tree_excluded_set = set(
    TREE_EXCLUDED_FEATURES
)

LOGISTIC_FEATURES = tuple(
    feature
    for feature in engineered_feature_order
    if feature not in logistic_excluded_set
)

TREE_FEATURES = tuple(
    feature
    for feature in engineered_feature_order
    if feature not in tree_excluded_set
)

len(engineered_feature_order), (
    len(LOGISTIC_FEATURES),
    len(TREE_FEATURES),
)

(134, (128, 129))

In [48]:
categorical_feature_set = set(
    RAW_CATEGORICAL_FEATURES
)

LOGISTIC_CATEGORICAL_FEATURES = tuple(
    feature
    for feature in LOGISTIC_FEATURES
    if feature in categorical_feature_set
)

LOGISTIC_NUMERICAL_FEATURES = tuple(
    feature
    for feature in LOGISTIC_FEATURES
    if feature not in categorical_feature_set
)

TREE_CATEGORICAL_FEATURES = tuple(
    feature
    for feature in TREE_FEATURES
    if feature in categorical_feature_set
)

TREE_NUMERICAL_FEATURES = tuple(
    feature
    for feature in TREE_FEATURES
    if feature not in categorical_feature_set
)

In [50]:
model_feature_view_summary = pd.DataFrame(
    {
        "total_features": {
            "logistic_regression": len(
                LOGISTIC_FEATURES
            ),
            "tree_based": len(
                TREE_FEATURES
            ),
        },
        "numerical_features": {
            "logistic_regression": len(
                LOGISTIC_NUMERICAL_FEATURES
            ),
            "tree_based": len(
                TREE_NUMERICAL_FEATURES
            ),
        },
        "categorical_features": {
            "logistic_regression": len(
                LOGISTIC_CATEGORICAL_FEATURES
            ),
            "tree_based": len(
                TREE_CATEGORICAL_FEATURES
            ),
        },
        "excluded_features": {
            "logistic_regression": len(
                LOGISTIC_EXCLUDED_FEATURES
            ),
            "tree_based": len(
                TREE_EXCLUDED_FEATURES
            ),
        },
    }
)

model_feature_view_summary

,total_features,numerical_features,categorical_features,excluded_features
logistic_regression,128,112,16,6
tree_based,129,113,16,5


In [51]:
class FeatureSubsetSelector(
    BaseEstimator,
    TransformerMixin,
):
    """
    Select an ordered subset of DataFrame columns.

    The selected feature list is fixed before model fitting
    and does not depend on the target.
    """

    def __init__(
        self,
        selected_features,
    ):
        self.selected_features = selected_features

    def _validate_dataframe(
        self,
        X: pd.DataFrame,
    ) -> None:
        if not isinstance(X, pd.DataFrame):
            raise TypeError(
                "FeatureSubsetSelector expects "
                "a pandas DataFrame."
            )

    def fit(
        self,
        X: pd.DataFrame,
        y=None,
    ):
        self._validate_dataframe(X)

        selected_features = tuple(
            self.selected_features
        )

        if len(selected_features) == 0:
            raise ValueError(
                "selected_features cannot be empty."
            )

        if (
            len(selected_features)
            != len(set(selected_features))
        ):
            raise ValueError(
                "selected_features contains duplicates."
            )

        missing_features = (
            set(selected_features)
            - set(X.columns)
        )

        if missing_features:
            raise ValueError(
                "Selected features are missing from "
                f"the input: {sorted(missing_features)}"
            )

        self.feature_names_in_ = np.asarray(
            X.columns,
            dtype=object,
        )

        self.n_features_in_ = len(
            self.feature_names_in_
        )

        self.selected_features_ = np.asarray(
            selected_features,
            dtype=object,
        )

        return self

    def transform(
        self,
        X: pd.DataFrame,
    ) -> pd.DataFrame:
        check_is_fitted(
            self,
            attributes=[
                "feature_names_in_",
                "selected_features_",
            ],
        )

        self._validate_dataframe(X)

        if not X.columns.equals(
            pd.Index(self.feature_names_in_)
        ):
            raise ValueError(
                "Input columns or their order differ "
                "from the fitted schema."
            )

        return (
            X.loc[:, self.selected_features_]
            .copy()
        )

    def get_feature_names_out(
        self,
        input_features=None,
    ) -> np.ndarray:
        check_is_fitted(
            self,
            attributes=["selected_features_"],
        )

        if input_features is not None:
            input_features = np.asarray(
                input_features,
                dtype=object,
            )

            if not np.array_equal(
                input_features,
                self.feature_names_in_,
            ):
                raise ValueError(
                    "input_features does not match "
                    "feature_names_in_."
                )

        return self.selected_features_.copy()

In [54]:
logistic_feature_selector = (
    FeatureSubsetSelector(
        selected_features=LOGISTIC_FEATURES,
    )
)

tree_feature_selector = (
    FeatureSubsetSelector(
        selected_features=TREE_FEATURES,
    )
)

X_development_logistic_view = (
    logistic_feature_selector.fit_transform(
        X_development_engineered
    )
)

X_development_tree_view = (
    tree_feature_selector.fit_transform(
        X_development_engineered
    )
)

(
    X_development_logistic_view.shape,
    X_development_tree_view.shape,
)

((246008, 128), (246008, 129))

In [55]:
duration_redundancy_checks = pd.Series(
    {
        "age_rescaling_matches": np.allclose(
            X_development_engineered[
                "AGE_YEARS"
            ],
            (
                -X_development_raw[
                    "DAYS_BIRTH"
                ]
                / 365.25
            ),
            equal_nan=True,
        ),
        "employment_rescaling_matches": (
            np.allclose(
                X_development_engineered[
                    "EMPLOYMENT_YEARS"
                ],
                (
                    -X_development_engineered[
                        "DAYS_EMPLOYED"
                    ]
                    / 365.25
                ),
                equal_nan=True,
            )
        ),
        "registration_rescaling_matches": (
            np.allclose(
                X_development_engineered[
                    "REGISTRATION_YEARS"
                ],
                (
                    -X_development_raw[
                        "DAYS_REGISTRATION"
                    ]
                    / 365.25
                ),
                equal_nan=True,
            )
        ),
        "id_publish_rescaling_matches": (
            np.allclose(
                X_development_engineered[
                    "ID_PUBLISH_YEARS"
                ],
                (
                    -X_development_raw[
                        "DAYS_ID_PUBLISH"
                    ]
                    / 365.25
                ),
                equal_nan=True,
            )
        ),
    },
    name="passed",
)

duration_redundancy_checks

age_rescaling_matches             True
employment_rescaling_matches      True
registration_rescaling_matches    True
id_publish_rescaling_matches      True
Name: passed, dtype: bool

In [56]:
ratio_pair = X_development_engineered[
    [
        "ANNUITY_CREDIT_RATIO",
        "CREDIT_TERM_PROXY",
    ]
]

complete_ratio_mask = (
    ratio_pair.notna().all(axis=1)
)

reciprocal_products = (
    ratio_pair.loc[
        complete_ratio_mask,
        "ANNUITY_CREDIT_RATIO",
    ]
    * ratio_pair.loc[
        complete_ratio_mask,
        "CREDIT_TERM_PROXY",
    ]
)

reciprocal_ratio_check = np.allclose(
    reciprocal_products,
    1.0,
    rtol=1e-10,
    atol=1e-10,
)

reciprocal_ratio_check

True

In [57]:
employment_start_age_check = np.allclose(
    X_development_engineered[
        "EMPLOYMENT_START_AGE"
    ],
    (
        X_development_engineered[
            "AGE_YEARS"
        ]
        - X_development_engineered[
            "EMPLOYMENT_YEARS"
        ]
    ),
    equal_nan=True,
)

employment_start_age_check

True

In [58]:
logistic_feature_set = set(
    LOGISTIC_FEATURES
)

tree_feature_set = set(
    TREE_FEATURES
)

logistic_numerical_set = set(
    LOGISTIC_NUMERICAL_FEATURES
)

logistic_categorical_set = set(
    LOGISTIC_CATEGORICAL_FEATURES
)

tree_numerical_set = set(
    TREE_NUMERICAL_FEATURES
)

tree_categorical_set = set(
    TREE_CATEGORICAL_FEATURES
)

feature_view_checks = pd.Series(
    {
        "logistic_expected_feature_count": (
            len(LOGISTIC_FEATURES) == 128
        ),
        "tree_expected_feature_count": (
            len(TREE_FEATURES) == 129
        ),
        "logistic_view_shape_matches": (
            X_development_logistic_view.shape[1]
            == len(LOGISTIC_FEATURES)
        ),
        "tree_view_shape_matches": (
            X_development_tree_view.shape[1]
            == len(TREE_FEATURES)
        ),
        "logistic_index_preserved": (
            X_development_logistic_view
            .index
            .equals(
                X_development_engineered.index
            )
        ),
        "tree_index_preserved": (
            X_development_tree_view
            .index
            .equals(
                X_development_engineered.index
            )
        ),
        "target_absent_from_logistic": (
            TARGET_COLUMN
            not in logistic_feature_set
        ),
        "identifier_absent_from_logistic": (
            IDENTIFIER_COLUMN
            not in logistic_feature_set
        ),
        "target_absent_from_tree": (
            TARGET_COLUMN
            not in tree_feature_set
        ),
        "identifier_absent_from_tree": (
            IDENTIFIER_COLUMN
            not in tree_feature_set
        ),
        "common_exclusions_absent_from_logistic": (
            set(COMMON_EXCLUDED_FEATURES)
            .isdisjoint(logistic_feature_set)
        ),
        "common_exclusions_absent_from_tree": (
            set(COMMON_EXCLUDED_FEATURES)
            .isdisjoint(tree_feature_set)
        ),
        "employment_start_age_absent_from_logistic": (
            "EMPLOYMENT_START_AGE"
            not in logistic_feature_set
        ),
        "employment_start_age_retained_for_tree": (
            "EMPLOYMENT_START_AGE"
            in tree_feature_set
        ),
        "model_view_difference_is_expected": (
            tree_feature_set
            - logistic_feature_set
            == {"EMPLOYMENT_START_AGE"}
        ),
        "logistic_partition_is_complete": (
            logistic_numerical_set
            | logistic_categorical_set
            == logistic_feature_set
        ),
        "logistic_partition_is_disjoint": (
            logistic_numerical_set
            .isdisjoint(
                logistic_categorical_set
            )
        ),
        "tree_partition_is_complete": (
            tree_numerical_set
            | tree_categorical_set
            == tree_feature_set
        ),
        "tree_partition_is_disjoint": (
            tree_numerical_set
            .isdisjoint(
                tree_categorical_set
            )
        ),
        "all_duration_checks_pass": (
            duration_redundancy_checks.all()
        ),
        "reciprocal_ratio_check_passes": (
            reciprocal_ratio_check
        ),
        "employment_identity_check_passes": (
            employment_start_age_check
        ),
    },
    name="passed",
)

feature_view_checks

logistic_expected_feature_count              True
tree_expected_feature_count                  True
logistic_view_shape_matches                  True
tree_view_shape_matches                      True
logistic_index_preserved                     True
tree_index_preserved                         True
target_absent_from_logistic                  True
identifier_absent_from_logistic              True
target_absent_from_tree                      True
identifier_absent_from_tree                  True
common_exclusions_absent_from_logistic       True
common_exclusions_absent_from_tree           True
employment_start_age_absent_from_logistic    True
employment_start_age_retained_for_tree       True
model_view_difference_is_expected            True
logistic_partition_is_complete               True
logistic_partition_is_disjoint               True
tree_partition_is_complete                   True
tree_partition_is_disjoint                   True
all_duration_checks_pass                     True


In [59]:
feature_order_checks = pd.Series(
    {
        "logistic_selector_order_matches": (
            tuple(
                X_development_logistic_view.columns
            )
            == LOGISTIC_FEATURES
        ),
        "tree_selector_order_matches": (
            tuple(
                X_development_tree_view.columns
            )
            == TREE_FEATURES
        ),
        "logistic_categorical_count": (
            len(
                LOGISTIC_CATEGORICAL_FEATURES
            )
            == 16
        ),
        "tree_categorical_count": (
            len(
                TREE_CATEGORICAL_FEATURES
            )
            == 16
        ),
    },
    name="passed",
)

feature_order_checks

logistic_selector_order_matches    True
tree_selector_order_matches        True
logistic_categorical_count         True
tree_categorical_count             True
Name: passed, dtype: bool

# 5.Leakage-safe preprocessing pipelines

This section defines separate preprocessing pipelines for logistic regression and
tree-based models.

The preprocessing strategy:

- imputes numerical variables with training-fold medians;
- preserves numerical missingness through explicit indicators;
- represents missing categorical values as a separate category;
- keeps existing values such as `XNA`, `Unknown` and `UNKNOWN` unchanged;
- one-hot encodes categorical variables for logistic regression;
- ordinally encodes categorical variables for the compact tree-based baseline;
- standardizes numerical variables only for logistic regression;
- handles categories that were not observed during fitting;
- performs every learned transformation inside the modeling pipeline.

The labeled holdout set remains isolated.

In [ ]:
from scipy import sparse

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
)


'1.6.1'

In [61]:
def make_numerical_preprocessor(
    scale: bool,
) -> Pipeline:
    """
    Build a numerical preprocessing pipeline.

    Medians and missing-value indicators are learned only
    when the pipeline is fitted.
    """
    steps = [
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
                keep_empty_features=True,
            ),
        ),
    ]

    if scale:
        steps.append(
            (
                "scaler",
                StandardScaler(),
            )
        )

    return Pipeline(steps=steps)

In [62]:
def make_logistic_categorical_preprocessor(
) -> Pipeline:
    """
    Impute and one-hot encode categorical variables.
    """
    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    drop=None,
                    sparse_output=True,
                    dtype=np.float64,
                ),
            ),
        ]
    )

In [63]:
def make_tree_categorical_preprocessor(
) -> Pipeline:
    """
    Impute and compactly encode categorical variables
    for the tree-based baseline.
    """
    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "encoder",
                OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=-1,
                    dtype=np.float64,
                ),
            ),
        ]
    )

In [64]:
def make_logistic_preprocessor(
) -> ColumnTransformer:
    """
    Build the preprocessing stage used by
    logistic-regression models.
    """
    return ColumnTransformer(
        transformers=[
            (
                "numerical",
                make_numerical_preprocessor(
                    scale=True,
                ),
                list(
                    LOGISTIC_NUMERICAL_FEATURES
                ),
            ),
            (
                "categorical",
                (
                    make_logistic_categorical_preprocessor()
                ),
                list(
                    LOGISTIC_CATEGORICAL_FEATURES
                ),
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=True,
    )

In [65]:
def make_tree_preprocessor(
) -> ColumnTransformer:
    """
    Build the compact preprocessing stage used by
    tree-based baseline models.
    """
    return ColumnTransformer(
        transformers=[
            (
                "numerical",
                make_numerical_preprocessor(
                    scale=False,
                ),
                list(
                    TREE_NUMERICAL_FEATURES
                ),
            ),
            (
                "categorical",
                (
                    make_tree_categorical_preprocessor()
                ),
                list(
                    TREE_CATEGORICAL_FEATURES
                ),
            ),
        ],
        remainder="drop",
        sparse_threshold=0.0,
        verbose_feature_names_out=True,
    )

In [66]:
def make_logistic_preprocessing_pipeline(
) -> Pipeline:
    return Pipeline(
        steps=[
            (
                "feature_engineering",
                ApplicationFeatureEngineer(),
            ),
            (
                "feature_selection",
                FeatureSubsetSelector(
                    selected_features=(
                        LOGISTIC_FEATURES
                    ),
                ),
            ),
            (
                "preprocessor",
                make_logistic_preprocessor(),
            ),
        ]
    )

In [67]:
def make_tree_preprocessing_pipeline(
) -> Pipeline:
    return Pipeline(
        steps=[
            (
                "feature_engineering",
                ApplicationFeatureEngineer(),
            ),
            (
                "feature_selection",
                FeatureSubsetSelector(
                    selected_features=(
                        TREE_FEATURES
                    ),
                ),
            ),
            (
                "preprocessor",
                make_tree_preprocessor(),
            ),
        ]
    )

In [68]:
logistic_preprocessing_pipeline = (
    make_logistic_preprocessing_pipeline()
)

tree_preprocessing_pipeline = (
    make_tree_preprocessing_pipeline()
)

In [69]:
(
    X_preprocessing_audit_train,
    X_preprocessing_audit_validation,
    y_preprocessing_audit_train,
    y_preprocessing_audit_validation,
) = train_test_split(
    X_development_raw,
    y_development,
    train_size=50_000,
    test_size=10_000,
    random_state=RANDOM_STATE,
    stratify=y_development,
)

logistic_audit_pipeline = clone(
    logistic_preprocessing_pipeline
)

tree_audit_pipeline = clone(
    tree_preprocessing_pipeline
)

logistic_audit_pipeline.fit(
    X_preprocessing_audit_train,
    y_preprocessing_audit_train,
)

tree_audit_pipeline.fit(
    X_preprocessing_audit_train,
    y_preprocessing_audit_train,
)

X_logistic_audit_transformed = (
    logistic_audit_pipeline.transform(
        X_preprocessing_audit_validation
    )
)

X_tree_audit_transformed = (
    tree_audit_pipeline.transform(
        X_preprocessing_audit_validation
    )
)

(
    X_logistic_audit_transformed.shape,
    X_tree_audit_transformed.shape,
)

((10000, 324), (10000, 196))

In [70]:
output_type_summary = pd.DataFrame(
    {
        "is_sparse": {
            "logistic": sparse.issparse(
                X_logistic_audit_transformed
            ),
            "tree_based": sparse.issparse(
                X_tree_audit_transformed
            ),
        },
        "rows": {
            "logistic": (
                X_logistic_audit_transformed.shape[0]
            ),
            "tree_based": (
                X_tree_audit_transformed.shape[0]
            ),
        },
        "columns": {
            "logistic": (
                X_logistic_audit_transformed.shape[1]
            ),
            "tree_based": (
                X_tree_audit_transformed.shape[1]
            ),
        },
    }
)

output_type_summary

,is_sparse,rows,columns
logistic,True,10000,324
tree_based,False,10000,196


In [72]:
logistic_output_features = (
    logistic_audit_pipeline
    .get_feature_names_out()
)

tree_output_features = (
    tree_audit_pipeline
    .get_feature_names_out()
)

(
    len(logistic_output_features),
    X_logistic_audit_transformed.shape[1],
)

(324, 324)

In [73]:
(
    len(tree_output_features),
    X_tree_audit_transformed.shape[1],
)

(196, 196)

In [77]:
def transformed_matrix_is_finite(
    matrix,
) -> bool:
    """
    Check finite stored values in a dense or sparse matrix.
    """
    if sparse.issparse(matrix):
        values = matrix.data
    else:
        values = np.asarray(matrix)

    return bool(
        np.isfinite(values).all()
    )

finite_value_checks = pd.Series(
    {
        "logistic_values_are_finite": (
            transformed_matrix_is_finite(
                X_logistic_audit_transformed
            )
        ),
        "tree_values_are_finite": (
            transformed_matrix_is_finite(
                X_tree_audit_transformed
            )
        ),
    },
    name="passed",
)

finite_value_checks

logistic_values_are_finite    True
tree_values_are_finite        True
Name: passed, dtype: bool

In [80]:
unknown_category_test = (
    X_preprocessing_audit_validation
    .head(1)
    .copy()
)

unknown_category_feature = (
    RAW_CATEGORICAL_FEATURES[0]
)

unknown_category_test.loc[
    :,
    unknown_category_feature,
] = "__UNSEEN_CATEGORY_FOR_TEST__"

logistic_unknown_output = (
    logistic_audit_pipeline.transform(
        unknown_category_test
    )
)

tree_unknown_output = (
    tree_audit_pipeline.transform(
        unknown_category_test
    )
)

unknown_category_checks = pd.Series(
    {
        "logistic_unknown_category_supported": (
            logistic_unknown_output.shape[1]
            == X_logistic_audit_transformed.shape[1]
        ),
        "tree_unknown_category_supported": (
            tree_unknown_output.shape[1]
            == X_tree_audit_transformed.shape[1]
        ),
        "logistic_unknown_output_is_finite": (
            transformed_matrix_is_finite(
                logistic_unknown_output
            )
        ),
        "tree_unknown_output_is_finite": (
            transformed_matrix_is_finite(
                tree_unknown_output
            )
        ),
    },
    name="passed",
)

unknown_category_checks

logistic_unknown_category_supported    True
tree_unknown_category_supported        True
logistic_unknown_output_is_finite      True
tree_unknown_output_is_finite          True
Name: passed, dtype: bool

In [84]:
logistic_fitted_preprocessor = (
    logistic_audit_pipeline
    .named_steps["preprocessor"]
)

logistic_numerical_imputer = (
    logistic_fitted_preprocessor
    .named_transformers_["numerical"]
    .named_steps["imputer"]
)

median_check_feature = "AMT_ANNUITY"

median_feature_position = (
    list(LOGISTIC_NUMERICAL_FEATURES)
    .index(median_check_feature)
)

learned_median = (
    logistic_numerical_imputer
    .statistics_[median_feature_position]
)

expected_training_median = (
    X_preprocessing_audit_train[
        median_check_feature
    ]
    .median()
)

learned_median, expected_training_median

(24939.0, 24939.0)

In [85]:
median_fit_check = np.isclose(
    learned_median,
    expected_training_median,
    equal_nan=True,
)

median_fit_check

True

In [86]:
preprocessing_checks = pd.Series(
    {
        "logistic_row_count_preserved": (
            X_logistic_audit_transformed.shape[0]
            == len(
                X_preprocessing_audit_validation
            )
        ),
        "tree_row_count_preserved": (
            X_tree_audit_transformed.shape[0]
            == len(
                X_preprocessing_audit_validation
            )
        ),
        "logistic_output_is_sparse": (
            sparse.issparse(
                X_logistic_audit_transformed
            )
        ),
        "tree_output_is_dense": (
            not sparse.issparse(
                X_tree_audit_transformed
            )
        ),
        "logistic_feature_names_match_shape": (
            len(logistic_output_features)
            == X_logistic_audit_transformed.shape[1]
        ),
        "tree_feature_names_match_shape": (
            len(tree_output_features)
            == X_tree_audit_transformed.shape[1]
        ),
        "logistic_values_are_finite": (
            finite_value_checks[
                "logistic_values_are_finite"
            ]
        ),
        "tree_values_are_finite": (
            finite_value_checks[
                "tree_values_are_finite"
            ]
        ),
        "unknown_categories_supported": (
            unknown_category_checks.all()
        ),
        "training_median_was_learned": (
            median_fit_check
        ),
        "target_absent_from_logistic_output": (
            all(
                TARGET_COLUMN not in feature
                for feature
                in logistic_output_features
            )
        ),
        "identifier_absent_from_logistic_output": (
            all(
                IDENTIFIER_COLUMN not in feature
                for feature
                in logistic_output_features
            )
        ),
        "target_absent_from_tree_output": (
            all(
                TARGET_COLUMN not in feature
                for feature
                in tree_output_features
            )
        ),
        "identifier_absent_from_tree_output": (
            all(
                IDENTIFIER_COLUMN not in feature
                for feature
                in tree_output_features
            )
        ),
    },
    name="passed",
)

preprocessing_checks

logistic_row_count_preserved              True
tree_row_count_preserved                  True
logistic_output_is_sparse                 True
tree_output_is_dense                      True
logistic_feature_names_match_shape        True
tree_feature_names_match_shape            True
logistic_values_are_finite                True
tree_values_are_finite                    True
unknown_categories_supported              True
training_median_was_learned               True
target_absent_from_logistic_output        True
identifier_absent_from_logistic_output    True
target_absent_from_tree_output            True
identifier_absent_from_tree_output        True
Name: passed, dtype: bool

## Preprocessing findings

- Numerical variables are median-imputed and accompanied by missing-value
  indicators.
- Numerical variables are standardized only in the logistic-regression pipeline.
- Actual categorical missing values are represented by a dedicated
  `__MISSING__` category.
- Existing values such as `XNA`, `Unknown` and `UNKNOWN` remain distinct and are
  not automatically treated as missing.
- Logistic regression uses sparse one-hot encoding without dropping a reference
  category.
- The compact tree baseline uses ordinal encoding, with unseen categories encoded
  as `-1`.
- Both pipelines support categories that were absent during fitting.
- All learned statistics, including medians, means, standard deviations and
  category mappings, are estimated only when the pipeline is fitted.
- No resampling, target encoding, target-dependent feature selection or holdout
  transformation was performed.
- The holdout set remains isolated.